In [0]:
from pyspark.sql import functions as F

EMPLOYEE_TABLE = "workspace.firmable_v2_silver.employees_clean"
ASSIGNMENT_TABLE = "workspace.firmable_v2_silver.manager_assignments"

EDGE_TABLE = "workspace.firmable_v2_silver.hierarchy_edges"
HIERARCHY_TABLE = "workspace.firmable_v2_silver.employee_hierarchy"

MAX_DEPTH = 20

employees = spark.table(EMPLOYEE_TABLE).select(
    "company_id",
    "employee_id",
    "company_name",
    "job_title_clean",
    "department",
    "business_unit",
    "team_id",
    "team_name",
    "inferred_seniority_level",
    "management_authority",
    "title_function"
)

assignments = spark.table(ASSIGNMENT_TABLE).select(
    "company_id",
    "employee_id",
    F.col("candidate_manager_id").alias("manager_id"),
    "candidate_score",
    "relationship_type",
    "assignment_status",
    "assignment_round"
)

base = employees.join(
    assignments,
    ["company_id", "employee_id"],
    "left"
).cache()

edges = base.filter(
    F.col("manager_id").isNotNull()
).select(
    "company_id",
    "manager_id",
    "employee_id",
    "candidate_score",
    "relationship_type",
    "assignment_round"
).cache()

manager_ids = employees.select(
    "company_id",
    F.col("employee_id").alias("manager_id")
).dropDuplicates()

invalid_managers = edges.join(
    F.broadcast(manager_ids),
    ["company_id", "manager_id"],
    "left_anti"
).count()

assert invalid_managers == 0, (
    f"Found {invalid_managers} invalid manager references."
)

# Root employees have no assigned manager.
visited = base.filter(
    F.col("manager_id").isNull()
).select(
    "company_id",
    "employee_id",
    F.col("employee_id").alias("root_employee_id"),
    F.lit(0).alias("hierarchy_depth")
).dropDuplicates().cache()

frontier = visited

# Traverse from managers to their direct reports.
for depth in range(1, MAX_DEPTH + 1):

    parent_frontier = frontier.select(
        "company_id",
        F.col("employee_id").alias("parent_id"),
        "root_employee_id",
        "hierarchy_depth"
    )

    children = edges.alias("e").join(
        parent_frontier.alias("p"),
        (
            F.col("e.company_id") == F.col("p.company_id")
        ) & (
            F.col("e.manager_id") == F.col("p.parent_id")
        ),
        "inner"
    ).select(
        F.col("e.company_id").alias("company_id"),
        F.col("e.employee_id").alias("employee_id"),
        F.col("p.root_employee_id").alias("root_employee_id"),
        (F.col("p.hierarchy_depth") + 1).alias("hierarchy_depth")
    ).join(
        visited.select("company_id", "employee_id"),
        ["company_id", "employee_id"],
        "left_anti"
    ).dropDuplicates().cache()

    if children.limit(1).count() == 0:
        break

    visited = visited.unionByName(children).cache()
    frontier = children

# Attach hierarchy information to all employees.
hierarchy = base.join(
    visited,
    ["company_id", "employee_id"],
    "left"
).withColumn(
    "is_root",
    F.col("manager_id").isNull()
).withColumn(
    "root_type",
    F.when(
        F.col("is_root") &
        (F.col("inferred_seniority_level") == 0),
        F.lit("CEO_ROOT")
    ).when(
        F.col("is_root"),
        F.lit("UNASSIGNED_ROOT")
    ).otherwise(F.lit("CHILD"))
)

unreachable = hierarchy.filter(
    F.col("root_employee_id").isNull()
).count()

assert unreachable == 0, (
    f"{unreachable} employees are unreachable. "
    "Possible cycle or hierarchy depth issue."
)

# Ensure every employee appears once.
duplicate_rows = (
    hierarchy.groupBy("company_id", "employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_rows == 0

# Publish.
edges.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(EDGE_TABLE)

hierarchy.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(HIERARCHY_TABLE)

print("HIERARCHY CONSTRUCTION COMPLETED")
print("Total employees:", hierarchy.count())
print(
    "Assigned employees:",
    hierarchy.filter(F.col("manager_id").isNotNull()).count()
)
print("Component roots:", hierarchy.filter(F.col("is_root")).count())
print(
    "Maximum hierarchy depth:",
    hierarchy.agg(F.max("hierarchy_depth")).first()[0]
)
print("Unreachable employees:", unreachable)
print("Duplicate rows:", duplicate_rows)

print("\nROOT TYPE DISTRIBUTION")
hierarchy.filter(F.col("is_root")).groupBy(
    "root_type"
).count().show(truncate=False)

print("\nHIERARCHY DEPTH DISTRIBUTION")
hierarchy.groupBy("hierarchy_depth").count().orderBy(
    "hierarchy_depth"
).show()

print("\nOUTPUT TABLE:", HIERARCHY_TABLE)
print("ALL CHECKS PASSED")